# Supervised Classification mit SkLearn

In [10]:
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

## Daten Laden

In [2]:
data = read_csv("https://tinyurl.com/mcit-titanic")

### Vertikalen Split in Input/Output

In [3]:
X = data[[
    'Pclass',
    'Sex',
    'Age',
    'SibSp',
    'Parch',
    'Embarked'
 ]]

y = data['Survived']

In [4]:
print(X.shape, y.shape)

(891, 6) (891,)


### Horizontaler Split in Train/Test

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2
)

In [6]:
print("Shape-Check Trainings-Daten: X-shape =", X_train.shape, "und y-shape =", y_train.shape)
print("Shape-Check Test-Daten: X-shape =", X_test.shape, "und y-shape =", y_test.shape)

Shape-Check Trainings-Daten: X-shape = (712, 6) und y-shape = (712,)
Shape-Check Test-Daten: X-shape = (179, 6) und y-shape = (179,)


## Preprocess

In [7]:
numerical_cols = make_column_selector(dtype_include=['number'])
categorical_cols = make_column_selector(dtype_exclude=['number'])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [8]:
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [9]:
print(X_train_processed.shape, y_train.shape)

(712, 10) (712,)


## Machine Learning

In [35]:
tree_estimator = DecisionTreeClassifier(max_depth=4, min_samples_split=10)
tree_estimator.fit(X_train_processed, y_train)
tree_predictions = tree_estimator.predict(X_test_processed)

In [36]:
bayes_estimator = GaussianNB()
bayes_estimator.fit(X_train_processed, y_train)
bayes_predictions = bayes_estimator.predict(X_test_processed)

In [37]:
knn_estimator = KNeighborsClassifier(n_neighbors=5)
knn_estimator.fit(X_train_processed, y_train)
knn_predictions = knn_estimator.predict(X_test_processed)

In [39]:
print("Tree Accuracy =", accuracy_score(y_test, tree_predictions))
print("Bayes Accuracy =", accuracy_score(y_test, bayes_predictions))
print("KNN Accuracy =", accuracy_score(y_test, knn_predictions))

Tree Accuracy = 0.770949720670391
Bayes Accuracy = 0.6536312849162011
KNN Accuracy = 0.7653631284916201


In [42]:
print("Tree Accuracy =", accuracy_score(y_train, tree_estimator.predict(X_train_processed)))
print("Bayes Accuracy =", accuracy_score(y_train, bayes_estimator.predict(X_train_processed)))
print("KNN Accuracy =", accuracy_score(y_train, knn_estimator.predict(X_train_processed)))

Tree Accuracy = 0.8553370786516854
Bayes Accuracy = 0.6657303370786517
KNN Accuracy = 0.8679775280898876
